# Step 12 — ADP-C Evaluator QLoRA Training
**Phase:** 4 — Model Training & Adapter Alignment

**Spec authority:** SPEC-400 §3.1, §5; REQ-400-020–025

**Base model:** `google/gemma-2-2b-it` (Gemma-2 2B Instruct)

**Prerequisites:** Step 11 complete (`finetuning/adp_c_evaluator/data/adp_c_train.jsonl` exists)

**Output adapter:** `finetuning/adp_c_evaluator/adp_c_final/`

---

## Purpose

This notebook fine-tunes the **ADP-C Evaluator adapter** on the synthetic
dataset produced in Step 11. ADP-C is the pipeline's quality-and-safety gate:
every response generated by ADP-A passes through ADP-C before reaching the
user, and ADP-C's verdict (`PASS`, `REGENERATE`, or `ESCALATE`) determines
whether the response is delivered, regenerated, or replaced with a crisis
escalation path (SPEC-300, SPEC-700).

The fine-tuning uses **QLoRA** (Quantisation-aware Low-Rank Adaptation):
a small set of low-rank weight matrices is injected alongside the frozen
base model and trained to specialise the model's output distribution for
structured verdict classification. Only the LoRA matrices — totalling roughly
3–4 million parameters — are updated during training; the 2B base weights
remain frozen throughout.

The resulting adapter is saved in PEFT format to `adp_c_final/`. This adapter
directory is the sole input to Step 13 (ADP-A data preparation), where it
runs as the rejection-sampling oracle.

## Why ADP-C Trains First

ADP-C v1 is trained on synthetic data (Step 11) before any of the other
adapters exist, because it serves as the **quality filter** for the training
data of every downstream adapter:

- **Step 13** loads `adp_c_final/` and runs every candidate empathy record
  through ADP-C. Records that receive a `PASS` verdict are kept; others are
  discarded. This ensures ADP-A is trained only on responses that already meet
  the pipeline's quality threshold.
- **Step 15** follows the same pattern for ADP-B safety data.
- **Step 17** closes the loop: once ADP-A and ADP-B are trained, their real
  outputs are scored by ADP-C v1 to build a second-generation evaluator (v2)
  calibrated against the actual models it will judge in production.

Training ADP-C first is therefore a prerequisite for the entire pipeline
bootstrap sequence.

## Training Strategy

Training uses `SFTTrainer` (TRL) with `SFTConfig` for configuration. Key
choices:

- **Loss masking:** Only the assistant-turn tokens contribute to the cross-entropy
  loss. Prompt tokens (system + user) are masked out, so the model learns to
  generate the correct verdict JSON — not to predict the prompt back to itself.
- **Chat template:** `tokenizer.apply_chat_template()` handles Gemma-2's native
  `<start_of_turn>` / `<end_of_turn>` markup. No manual template string is
  constructed.
- **Stratified 90/10 split:** The dataset is split by verdict class to preserve
  class balance in both train and validation sets. Standard `train_test_split`
  is not used; a manual stratification loop ensures minority classes are
  represented in both halves.
- **Best-model checkpoint:** `load_best_model_at_end=True` is set, so the adapter
  saved at the end of training corresponds to the lowest validation loss
  checkpoint, not the final epoch.
- **Gradient flow:** `model.enable_input_require_grads()` is called before LoRA
  injection. This is required when training LoRA adapters on top of a frozen
  bf16 base without using `prepare_model_for_kbit_training()` — the latter is
  only needed for NF4/int8 quantised bases.

## Hardware Note (RTX 3070 — 8 GB VRAM)

Gemma-2-2b-it loads in **~4.0 GB bf16** on the RTX 3070, leaving approximately
4 GB free for the LoRA parameter gradients and optimiser state. At the chosen
LoRA rank (r=16), the adapter adds roughly 3–4 M trainable parameters —
well within the remaining VRAM budget. Expected step time: **6–8 seconds**.

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is set in Cell 01 to
prevent WDDM memory fragmentation from triggering out-of-memory errors on
Windows when the allocator needs to grow across segment boundaries.

The `max_memory` cap `{0: "5000MiB", "cpu": "16GiB"}` is set explicitly when
loading the base model to prevent PyTorch from reserving the full 8 GB and
leaving no room for training allocations.

---

## Architecture Notes

| Decision | Choice made | Why not the alternative |
|----------|-------------|------------------------|
| Base model size | 2B (Gemma-2-2b-it) | A 7B base requires ~14 GB bf16 — exceeds RTX 3070 VRAM; training time per step rises from ~6s to ~37s |
| Quantisation | None (native bf16) | NF4 4-bit reduces footprint but introduces a second memory spike for full-precision gradient accumulation; at 2B, bf16 fits natively without the complexity |
| LoRA rank | 16 | Rank 8 converged more slowly in preliminary tests; rank 32 exceeded VRAM for step batch >1; rank 16 is the reliable midpoint at this scale |
| `enable_input_require_grads()` | Required | `prepare_model_for_kbit_training()` (the Hugging Face convenience alternative) is only needed when the base is loaded in kbit (NF4/int8); calling it on a bf16 base is a no-op at best and can introduce unexpected behaviour |
| SFTTrainer vs manual training loop | SFTTrainer | Manual loop requires explicit loss masking, gradient clipping, scheduler management; SFTTrainer handles all of this with `dataset_text_field` and `packing=False` |


In [1]:
# ── Cell 0: Pre-flight ────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Import datasets and trl BEFORE torch initialises the CUDA context.
# On Windows, importing datasets after CUDA is active triggers a pyarrow
# multiprocessing conflict that segfaults the kernel. Importing first, in a
# CUDA-free state, matches the clean terminal environment where both work fine.
import datasets
import trl

import subprocess
import torch

assert torch.cuda.is_available(), "CUDA not available — run finetuning/test_env.py."

total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Device: {torch.cuda.get_device_name(0)}  |  Total VRAM: {total_gb:.1f} GB")

try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()
    used_mb, free_mb = [int(x.strip()) for x in smi.split(",")]
    print(f"nvidia-smi — Used: {used_mb} MiB  |  Free: {free_mb} MiB")
    # Gemma-2-2b-it in bf16 occupies ~4 GB. Recommend >= 5000 MiB free to leave
    # headroom for activations and gradient buffers during LoRA training.
    if free_mb < 5000:
        print(f"\nWARN: Only {free_mb} MiB free. Close other GPU apps first.")
    else:
        print(f"\nOK: {free_mb} MiB available. Safe to proceed.")
except Exception:
    print("nvidia-smi unavailable — proceeding with PyTorch estimate.")

print(f"\ndatasets : {datasets.__version__}")
print(f"trl      : {trl.__version__}")

C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: NVIDIA GeForce RTX 3070  |  Total VRAM: 8.0 GB
nvidia-smi — Used: 700 MiB  |  Free: 7319 MiB

OK: 7319 MiB available. Safe to proceed.

datasets : 3.1.0
trl      : 0.11.4


In [2]:
# ── Cell 0b: Package check ────────────────────────────────────────────────────
# trl and datasets must be installed before running this notebook:
#   conda activate nikko && pip install trl datasets
# This cell confirms they are present and fails fast if not.

import trl, datasets
print(f"  trl     : {trl.__version__}")
print(f"  datasets: {datasets.__version__}")

  trl     : 0.11.4
  datasets: 3.1.0


In [3]:
# ── Cell 1: Safe imports ───────────────────────────────────────────────────────
# Only standard library and packages confirmed safe at import time.
# trl and datasets are imported lazily in the cells that use them — both have
# been observed to crash Jupyter kernels on Windows during module initialisation
# due to multiprocessing (datasets/pyarrow) and CUDA extension loading (trl).
import os
import gc
import json
import re
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

# Suppress tokenizer parallelism — prevents a known Windows multiprocessing
# deadlock when datasets/tokenizers spawns worker processes in a Jupyter kernel.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Safe imports OK.")

Safe imports OK.


## Section 1 — Training Configuration

All parameters trace to `finetuning/adp_c_evaluator/config.yaml` and SPEC-400 §3.1.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `lora_r` | 8 | ~2 M trainable params (0.09 % of 2 B). Sufficient for JSON classification. |
| `max_seq_length` | 512 | Verdict JSON fits comfortably; covers full instruction + output. |
| `batch_size × grad_accum` | 4 × 8 = **32** | Stable gradient estimate on small dataset without OOM. |
| `num_epochs` | 10 | Gemma-2 at 2 B converges faster than Mistral-7B — fewer epochs needed. |
| Target train loss | **< 0.40** | Classification task with 3-class label space. |

In [4]:
BASE_MODEL_ID = "google/gemma-2-2b-it"

BASE_DIR       = Path(r"D:/Git Repos/nikko-companion")
FINETUNING     = BASE_DIR / "finetuning"
TRAIN_DATA     = FINETUNING / "adp_c_evaluator" / "data" / "adp_c_train.jsonl"
CHECKPOINT_DIR = FINETUNING / "adp_c_evaluator" / "checkpoints"
OUTPUT_DIR     = FINETUNING / "adp_c_evaluator" / "adp_c_final"

assert TRAIN_DATA.exists(), f"Training data not found: {TRAIN_DATA} — run Step 11 first."
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
# Gemma-2-2b-it is much smaller — fewer epochs needed to converge.
# No quantisation → larger effective batch possible without OOM.
NUM_EPOCHS     = 25
BATCH_SIZE     = 2       # doubled vs Mistral — VRAM headroom allows it
GRAD_ACCUM     = 4       
MAX_SEQ_LENGTH = 512
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 0.01

# ── LoRA config ───────────────────────────────────────────────────────────────
# r=8 sufficient for a classification-style evaluator (3 output classes).
# Gemma-2 attention uses q_proj / k_proj / v_proj / o_proj same as Mistral.
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ["q_proj", "k_proj", "v_proj", "o_proj"]

print("Configuration loaded.")
print(f"  Base model : {BASE_MODEL_ID}")
print(f"  Output dir : {OUTPUT_DIR}")
print(f"  Epochs     : {NUM_EPOCHS}  |  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

Configuration loaded.
  Base model : google/gemma-2-2b-it
  Output dir : D:\Git Repos\nikko-companion\finetuning\adp_c_evaluator\adp_c_final
  Epochs     : 25  |  Effective batch: 8


## Section 2 — Dataset Loading & Formatting

The ADP-C corpus from Step 11 (`adp_c_train.jsonl`) contains records with an
`instruction` (the audit prompt) and an `output` (JSON verdict).

Each record is formatted using Gemma-2's **native `apply_chat_template()`** rather
than Mistral's manual `[INST]...[/INST]` wrapping. The tokenizer inserts
`<start_of_turn>` / `<end_of_turn>` markers consistent with Gemma-2's pre-training.

A stratified 90/10 train/eval split preserves the verdict distribution in both sets.

In [5]:
# ── Load dataset from JSONL ───────────────────────────────────────────────────
assert TRAIN_DATA.exists(), (
    f"Dataset not found at {TRAIN_DATA}. "
    "Run step11_adp_c_data_generation.ipynb first."
)

raw_records = [
    json.loads(line)
    for line in TRAIN_DATA.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

from collections import Counter
verdicts = [json.loads(r["output"])["verdict"] for r in raw_records]
print(f"Loaded {len(raw_records)} records.")
print("\nVerdict distribution:")
for v, c in sorted(Counter(verdicts).items()):
    print(f"  {v:<12} {c:>4}  ({c/len(verdicts)*100:.1f}%)")

# ── Format using Gemma-2 chat template ────────────────────────────────────────

# [CONCEPT] Chat template
# Unlike Mistral's manual [INST]...[/INST] wrapping, Gemma-2 has a built-in
# chat template. apply_chat_template() inserts <start_of_turn> markers and the
# <bos> token automatically — consistent with the model's pre-training format.
# SFTTrainer uses the model-turn boundary to set loss weights correctly.

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_record(record: dict) -> dict:
    messages = [
        {"role": "user",      "content": record["instruction"]},
        {"role": "assistant", "content": record["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

# ── Stratified 90/10 train/eval split ─────────────────────────────────────────
# Stratify by verdict label so each split has the same class distribution.
import random
random.seed(42)
from collections import defaultdict
from datasets import Dataset

buckets = defaultdict(list)
for r in raw_records:
    buckets[json.loads(r["output"])["verdict"]].append(r)

train_records, eval_records = [], []
for label, items in buckets.items():
    random.shuffle(items)
    n_eval = max(1, int(len(items) * 0.10))
    eval_records.extend(items[:n_eval])
    train_records.extend(items[n_eval:])

random.shuffle(train_records)
random.shuffle(eval_records)

train_data = Dataset.from_list([format_record(r) for r in train_records])
eval_data  = Dataset.from_list([format_record(r) for r in eval_records])

print(f"\nTrain: {len(train_data)}  |  Eval: {len(eval_data)}")

# Rough length check — flag records approaching the seq limit.
threshold  = MAX_SEQ_LENGTH * 4 * 0.8
long_count = sum(1 for r in train_data if len(r["text"]) > threshold)
if long_count > 0:
    print(f"Records near seq limit (>{int(threshold)} chars): {long_count} — will be truncated.")

Loaded 166 records.

Verdict distribution:
  APPROVE        43  (25.9%)
  REGENERATE     47  (28.3%)
  REJECT         76  (45.8%)

Train: 151  |  Eval: 15


## Section 3 — Model Loading (bf16, no quantisation)

Gemma-2-2b-it in bf16 occupies ~4.0 GB VRAM — well within the RTX 3070's 8 GB.
Two mechanisms keep the training footprint under the hardware limit:

1. **`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`** (Cell 0) — prevents
   Windows WDDM memory fragmentation from producing spurious OOM errors during
   gradient accumulation.

2. **`torch_dtype=torch.bfloat16`** — loads in 16-bit brain float instead of
   the default fp32. Halves VRAM vs full precision. No NF4 quantisation required
   at 2 B scale — bf16 is numerically stable for LoRA training without it.

Compare with Mistral Step 12: that notebook needed `BitsAndBytesConfig` (NF4 4-bit)
to fit 7 B into 8 GB. At 2 B, bf16 gives the same training quality with no quantisation
overhead and removes the `bitsandbytes` Windows crash risk entirely.

In [6]:
# ── Clear residual state before model load ───────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

before_mib = torch.cuda.memory_allocated(0) / 1024**2
print(f"Allocated VRAM before load: {before_mib:.0f} MiB")

# ── Load Gemma-2-2b-it in bf16 ────────────────────────────────────────────────

# [CONCEPT] bf16 vs NF4 quantisation
# Mistral-7B (7 B) required NF4 4-bit quantisation to fit in 8 GB — halving its
# 14 GB fp16 footprint to ~3.5 GB via BitsAndBytesConfig. Gemma-2-2b-it (2 B)
# needs only ~4 GB in bf16, so no quantisation is required. This removes the
# bitsandbytes dependency and its Windows ZeroGPU crash risk entirely.
#
# bf16 (bfloat16) vs fp16: both are 16-bit, but bf16 has a wider exponent range
# (same as fp32), making it numerically stable for training without loss scaling.
# fp16's narrow exponent can produce NaN gradients on deep models — bf16 avoids this.

print(f"Loading {BASE_MODEL_ID} in bf16...")
print(f"  Tip: first run downloads ~4.8 GB from HuggingFace Hub (subsequent runs use cache).")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# padding_side="right" required: SFTTrainer's loss masking scans left-to-right.
# Left-padding would misalign the [/INST] boundary and corrupt the loss mask.
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    max_memory={0: "5500MiB", "cpu": "16GiB"},   
)
model.config.use_cache = False  # Disabled during training (incompatible with grad checkpointing).

after_mib  = torch.cuda.memory_allocated(0) / 1024**2
peak_mib   = torch.cuda.max_memory_allocated(0) / 1024**2
print(f"\nModel loaded.")
print(f"  Allocated VRAM : {after_mib:.0f} MiB  (~{after_mib/1024:.1f} GB)")
print(f"  Peak VRAM      : {peak_mib:.0f} MiB  (includes load-time spike)")

Allocated VRAM before load: 0 MiB
Loading google/gemma-2-2b-it in bf16...
  Tip: first run downloads ~4.8 GB from HuggingFace Hub (subsequent runs use cache).


C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\accelerate\utils\modeling.py:329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  new_value = value.to(device)
Loading checkpoint shards: 100%|█████████████████| 2/2 [00:03<00:00,  1.79s/it]



Model loaded.
  Allocated VRAM : 4987 MiB  (~4.9 GB)
  Peak VRAM      : 4987 MiB  (includes load-time spike)


## Section 4 — LoRA Injection

With the base model loaded and all weights frozen, this section injects
trainable LoRA matrices and enables gradient flow through the adapter.

At r=8 across 4 projections (q/k/v/o) in each of Gemma-2's 26 transformer layers:
~2 M trainable parameters (0.09% of 2 B total). Sufficient to teach
structured JSON classification without requiring the full model to be updated.

In [7]:
# ── LoRA injection ────────────────────────────────────────────────────────────

# [CONCEPT] LoRA rank and parameter count
# LoRA injects matrices A (d × r) and B (r × d) into each target attention
# projection. The effective weight update is B @ A, scaled by alpha/r.
# At r=8 across 4 projections × 26 Gemma-2 layers: ~2 M trainable params (0.09%).
# Sufficient for teaching structured JSON classification without full fine-tuning.

# [CONCEPT] bf16 vs kbit training
# Unlike the Mistral notebook, we do NOT call prepare_model_for_kbit_training() here.
# That function is only needed when the base is quantised (NF4/int8) — it upcasts
# layer norms to fp32 for numerical stability. Our Gemma-2 base is already in bf16,
# so the layer norms are stable and no upcast is required.

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGETS,
)
model = get_peft_model(model, lora_config)

# enable_input_require_grads() is required when the base model's embedding
# layer is frozen but we still need gradients to flow back through it to the
# LoRA adapter layers. Without this, the backward pass terminates at the
# embedding and adapters receive no gradient signal.
model.enable_input_require_grads()

model.print_trainable_parameters()
print(f"VRAM after LoRA injection: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438
VRAM after LoRA injection: 4.89 GB


## Section 5 — Training Setup

SFTTrainer handles the training loop, gradient accumulation, evaluation, and checkpointing.
Expected training time on RTX 3070: **5–15 minutes** depending on dataset size.
Target: final train loss < 0.40. If loss is still > 0.60 after epoch 3,
increase `NUM_EPOCHS` to 15 and check data quality in Step 11.

In [8]:
# ── SFT configuration ─────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# [CONCEPT] SFTConfig and load_best_model_at_end
# load_best_model_at_end=True reloads the checkpoint with the lowest eval_loss
# after training finishes — saves having to manually identify the best epoch.
# Constraint: eval_steps must equal save_steps (or be a divisor of it) so the
# evaluator runs at the same step the checkpoint is saved. Here both = 20.

sft_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    save_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    eval_strategy="steps",
    eval_steps=10,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
    data_seed=42,
)
print("SFTConfig set.")

# [CONCEPT] SFTTrainer loss masking
# SFTTrainer locates the model-turn boundary in each formatted example and sets
# cross-entropy loss weight to 0 for all tokens before it (the instruction prompt).
# Only the assistant-output tokens (JSON verdict) contribute to the cross-entropy.
# The model trains to generate verdicts, not to echo prompts.
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=sft_config,
)

steps_per_epoch = max(1, len(train_data) // (BATCH_SIZE * GRAD_ACCUM))
print("SFTTrainer ready.")
print(f"  Train samples    : {len(train_data)}")
print(f"  Eval samples     : {len(eval_data)}")
print(f"  Steps per epoch  : {steps_per_epoch}")
print(f"  Total steps      : {steps_per_epoch * NUM_EPOCHS}")
print(f"  VRAM before train: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

SFTConfig set.


Map: 100%|████████████████████████████| 15/15 [00:00<00:00, 2496.71 examples/s]

SFTTrainer ready.
  Train samples    : 151
  Eval samples     : 15
  Steps per epoch  : 18
  Total steps      : 450
  VRAM before train: 4.89 GB



C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\trl\trainer\sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


## Section 6 — Run Training

`trainer.train()` executes the full loop. Progress logs appear every 5 steps.
The best checkpoint (lowest eval loss) is automatically loaded at the end.

In [9]:
# use_cache and gradient_checkpointing are mutually incompatible.
# Gradient checkpointing discards and recomputes activations on the backward
# pass; the KV cache would try to store those same values, creating a conflict.
model.config.use_cache = False

print("Starting ADP-C training (Gemma-2-2b-it)...")
train_result = trainer.train()

print("\n── Training complete ──────────────────────────────────────────")
print(f"  Final train loss : {train_result.training_loss:.4f}")
print(f"  Total steps      : {train_result.global_step}")
runtime = train_result.metrics.get("train_runtime", 0)
print(f"  Runtime          : {runtime/60:.1f} min")
print(f"  Peak VRAM        : {torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GB")

if train_result.training_loss > 0.40:
    print("  ⚠ Loss above 0.40 — consider increasing NUM_EPOCHS or checking data quality.")
else:
    print("  ✓ Loss within acceptance threshold.")

Starting ADP-C training (Gemma-2-2b-it)...


Step,Training Loss,Validation Loss
10,4.756100,4.547839
20,3.310500,2.961712
30,1.868700,1.608514
40,1.209600,1.145188
50,0.892300,0.991995
60,0.844800,0.924565
70,0.774200,0.882870
80,0.653600,0.833428
90,0.643100,0.804492
100,0.554100,0.794476



── Training complete ──────────────────────────────────────────
  Final train loss : 0.4825
  Total steps      : 475
  Runtime          : 17.9 min
  Peak VRAM        : 6.15 GB
  ⚠ Loss above 0.40 — consider increasing NUM_EPOCHS or checking data quality.


## Section 7 — Save Adapter

Only the LoRA adapter weights (~15–20 MB for r=8) are saved — not the 4 GB base model.
At inference time: load the Gemma-2 base once, then load the adapter via
`PeftModel.from_pretrained(base, adapter_dir)`.

ADP-C shares `google/gemma-2-2b-it` with ADP-B. The HF Space loads the base
model once and hot-swaps adapters using `set_adapter()` — zero extra VRAM cost.

In [10]:
# ── Save adapter only — not the full base model ────────────────────────────────

# [CONCEPT] PEFT adapter saving
# save_pretrained() on a PEFT model writes only:
#   adapter_model.safetensors  — LoRA A and B matrices (~15–25 MB for r=8)
#   adapter_config.json        — lora_r, target_modules, base model name
# The base model is NOT re-saved; it stays on HuggingFace Hub.
# Load at inference: PeftModel.from_pretrained(base_model, adapter_dir)

trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

adapter_mb = sum(f.stat().st_size for f in OUTPUT_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Adapter saved  : {OUTPUT_DIR.resolve()}")
print(f"Adapter size   : {adapter_mb:.1f} MB")
print("Adapter checkpoint complete.")

***** train metrics *****
  epoch                    =       25.0
  total_flos               =  4959386GF
  train_loss               =     0.4825
  train_runtime            = 0:17:52.04
  train_samples_per_second =      3.521
  train_steps_per_second   =      0.443
Adapter saved  : D:\Git Repos\nikko-companion\finetuning\adp_c_evaluator\adp_c_final
Adapter size   : 64.2 MB
Adapter checkpoint complete.


## Section 8 — Smoke Test
Three checks verify ADP-C correctly classifies the three verdict types.

| Test | Input | Pass condition |
|------|-------|----------------|
| T1 | Clean empathic response | Verdict = APPROVE |
| T2 | Diagnostic claim | Verdict = REJECT |
| T3 | Directive advice | Verdict = REGENERATE or REJECT |

In [12]:
model.config.use_cache = True
model.gradient_checkpointing_disable()

def evaluate(user_msg: str, nikko_resp: str, mode: str = "comfort") -> str:
    # Match the training instruction format from make_record() exactly
    instruction = (
        f"You are ADP-C, the Nikko response evaluator. "
        f"Given the conversational mode and the candidate response below, "
        f"return a JSON verdict.\n\n"
        f"[MODE: {mode.upper()}]\n"
        f"Nikko response: {nikko_resp}"
    )
    messages = [{"role": "user", "content": instruction}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = trainer.model.generate(
            **inputs,
            max_new_tokens=150,   # was 80 — JSON reason strings need headroom
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

t1 = evaluate("I've been feeling anxious.", "That sounds really hard. I'm here to listen.", mode="comfort")
t2 = evaluate("I feel sad.", "You are experiencing clinical depression. See a psychiatrist.", mode="guidance")
t3 = evaluate("I can't sleep.", "You should take melatonin supplements every night.", mode="guidance")

import json as _json
def verdict(raw):
    try: return _json.loads(raw).get("verdict", "PARSE_ERROR")
    except: return "PARSE_ERROR"

p1 = verdict(t1) == "APPROVE"
p2 = verdict(t2) == "REJECT"
p3 = verdict(t3) in ("REGENERATE", "REJECT")

print("── Smoke test ─────────────────────────────────────────────────")
print(f"  T1 (APPROVE): {'PASS' if p1 else 'FAIL'}  →  {verdict(t1)}")
print(f"  T2 (REJECT) : {'PASS' if p2 else 'FAIL'}  →  {verdict(t2)}")
print(f"  T3 (REGEN)  : {'PASS' if p3 else 'FAIL'}  →  {verdict(t3)}")
if p1 and p2 and p3:
    print("\n✓ All smoke tests passed. ADP-C (Gemma-2) ready.")
    print("  Next: run Step 13 (ADP-A data preparation — uses ADP-C as oracle).")
else:
    print("\n⚠ One or more tests failed.")

── Smoke test ─────────────────────────────────────────────────
  T1 (APPROVE): PASS  →  APPROVE
  T2 (REJECT) : PASS  →  REJECT
  T3 (REGEN)  : PASS  →  REJECT

✓ All smoke tests passed. ADP-C (Gemma-2) ready.
  Next: run Step 13 (ADP-A data preparation — uses ADP-C as oracle).
